<a href="https://colab.research.google.com/github/esrakirbas/msc-thesis-reducing-hallucinations-in-medical-diagnosis/blob/main/experiments/experiment01_medqa_manual_grounding/Medical_Grounding_002_Experiment1_MedQA_Manual_Grounding_with_QWEN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U  transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 43.7 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [2]:
#Before running this notebook : Runtime>Change Runtime Type>T4 GPU
#PIPELINE STEP - 1 : SETUP (LOAD LIBS AND MODEL)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import json
# Check GPU
print("GPU Available : ", torch.cuda.is_available())
print("GPU Name : ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
#
# Login to Hugging Face with HF_TOKEN in secrets, no code necessary
#
# Load QWEN and tokenizer
model_name = "Qwen/Qwen2.5-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
#
print(f"model.dtype = {model.dtype}")
print(f"model.device = {model.device}")

GPU Available :  True
GPU Name :  Tesla T4


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

... (2 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.dtype = torch.float16
model.device = cuda:0


In [4]:
#PIPELINE STEP - 2 : LOAD DATA (READY JSON FILE)
#Load questions and corpus
with open("experiment01_questions_and_corpus.json", "r", encoding="utf-8") as file:
    experiment_data = json.load(file)

In [27]:
#PIPELINE STEP - 3 : INFERENCE - BASELINE
#For each question create a prompt and collect the responses
baseline_responses = []
for rec in experiment_data:
  prompt = "Question Index " + rec["question_index"] + " : "
  prompt += rec["question"]
  prompt += "\n"
  for option, value in rec["options"].items():
    prompt += option + ". " + value + "\n"
  #Ask the first question
  messages = [
      {"role": "system", "content": "Select the best answer from the given options.\n"+
                                    "Start your response with 'Answer:' followed by the selected option letter and its text.\n"
                                    "On the next line, write 'Explanation:' followed by a brief explanation." },
      {"role": "user", "content": prompt}
  ]
  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

  generated_ids = model.generate(
      **model_inputs,
      max_new_tokens=512,
      do_sample=False
  )
  generated_ids = [
      output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
  ]

  response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
  baseline_responses.append({"question_index" : rec["question_index"],
                             "response" : response })
print(baseline_responses)

[{'question_index': '2', 'response': "Answer: B. Cholesterol embolization\nExplanation: The patient's presentation with mottled, reticulated purplish discoloration of the feet, renal failure, and the presence of intravascular spindle-shaped vacuoles in the renal biopsy strongly suggest cholesterol embolization. This condition occurs when cholesterol crystals are dislodged from plaque in arteries and travel to the kidneys, causing ischemia and necrosis. The elevated creatinine levels and the presence of cholesterol crystals in the renal biopsy further support this diagnosis."}, {'question_index': '6', 'response': "Answer: A. Renal artery stenosis\nExplanation: The patient's symptoms of flank pain, abdominal tenderness, and findings on ultrasound (dilation of the right ureter and renal pelvis) are consistent with obstruction in the urinary tract. Given his history of diabetes and peripheral artery disease, which can lead to renal artery stenosis, this is the most plausible cause of the u

In [25]:
#PIPELINE STEP - 4 : INFERENCE - DIRECT CITATION ANCHORING
#For each question create a prompt and collect the responses
grounded_responses = []
for rec in experiment_data:
  prompt = "Context : " + rec["grounding_context"] + "\n"
  prompt += "Question Index " + rec["question_index"] + " : "
  prompt += rec["question"]
  prompt += "\n"
  for option, value in rec["options"].items():
    prompt += option + ". " + value + "\n"
  #Ask the first question
  messages = [
      {"role": "system", "content": "Select the best answer from the given options using only the provided context.\n" +
                                    "Start your response with 'Answer:' followed by the selected option letter and its text.\n"
                                    "On the next line, write 'Explanation:' followed by a brief explanation.\n"
                                    "On the next line, write 'Supporting Quotation:' followed by an exact quotation copied from the provided context.\n"
                                    "If the context does not provide enough evidence, state that clearly."},
      {"role": "user", "content": prompt}
  ]
  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

  generated_ids = model.generate(
      **model_inputs,
      max_new_tokens=512,
      do_sample=False
  )
  generated_ids = [
      output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
  ]

  response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
  grounded_responses.append({"question_index" : rec["question_index"],
                             "response" : response })
print(grounded_responses)

[{'question_index': '2', 'response': 'Answer: B. Cholesterol embolization\nExplanation: The patient presents with mottled, reticulated purplish discoloration of the feet, which is characteristic of cholesterol embolization syndrome. This condition occurs when emboli composed of cholesterol and other debris embolize from atherosclerotic plaques, leading to ischemia in the extremities.\nSupporting Quotation: "Atheroemboli to limbs tend to lodge in the small vessels of the muscle and skin and may not occlude the large vessels, distal pulses usually remain palpable."'}, {'question_index': '6', 'response': 'Answer: D. Urethral stricture\nExplanation: The patient presents with right flank pain, abdominal tenderness, and an ultrasound showing dilation of the right ureter and renal pelvis, which are consistent with urinary tract obstruction. Given his history of diabetes and peripheral artery disease, a urethral stricture is a plausible cause of obstruction.\nSupporting Quotation: "Common form

In [ ]:
baseline_correct_answers = 0
grounded_correct_answers = 0
for rec, baseline, grounded in zip(experiment_data, baseline_responses, grounded_responses):
    correct_answer = rec['answer']
    correct_option = next((option for option, value in rec["options"].items() if value == correct_answer), None)
    temp = baseline['response'].find("Answer: ") + 8
    baseline_answer = baseline['response'][temp:temp+1]
    temp = grounded['response'].find("Answer: ") + 8
    grounded_answer = grounded['response'][temp:temp+1]
    print(f"Question Index: {rec['question_index']}")
    print(f"Correct Answer : {correct_option}. {correct_answer}")
    print(f"Baseline Answer: {baseline_answer} - {'Correct' if baseline_answer == correct_option else 'Incorrect'}")
    print(f"Grounded Answer: {grounded_answer} - {'Correct' if grounded_answer == correct_option else 'Incorrect'}")
    print("-" * 50)
    if(baseline_answer == correct_option):
      baseline_correct_answers += 1
    if(grounded_answer == correct_option):
      grounded_correct_answers += 1
print(f"Baseline Correct Answers: {baseline_correct_answers}/{len(experiment_data)}")
print(f"Grounded Correct Answers: {grounded_correct_answers}/{len(experiment_data)}")
print("-" * 50)